# Simulation concepts with synthetic prices

This replaces the exploratory `Untitled3.ipynb`. These small examples use made-up prices to explain the implementation; they are not evidence of investment performance. Install the notebook dependencies from [README.md](README.md) and run all cells.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from MCForecastTools import MCSimulation

dates = pd.date_range("2020-01-01", periods=10, freq="B")
flat_prices = pd.DataFrame({"EXAMPLE": np.full(10, 100.0)}, index=dates)
flat_model = MCSimulation(flat_prices, num_simulation=50, num_trading_days=20, seed=42)
flat_paths = flat_model.calc_cumulative_return()
assert (flat_paths == 1).all().all()
print("Flat historical prices produce zero fitted drift and variance: every scenario stays at 1.")

## A seed controls sampling

A seed makes a run reproducible with the same input, settings, and numerical environment. It does not improve the model's validity. This class uses a local generator and does not reset NumPy's global random state.

In [ ]:
moves = np.array([0.01, -0.02, 0.03, 0.01, -0.01, 0.02, -0.03, 0.01, 0.02])
values = np.r_[100.0, 100 * np.exp(np.cumsum(moves))]
prices = pd.DataFrame({"EXAMPLE": values}, index=dates)
model = MCSimulation(prices, num_simulation=50, num_trading_days=20, seed=42)
first = model.calc_cumulative_return().copy()
second = model.calc_cumulative_return()
pd.testing.assert_frame_equal(first, second)
display(model.summarize_cumulative_return())

## Growth factor, return, and terminal value

A growth factor of 1.12 means a 12% return: `1.12 - 1 = 0.12`. For 2,500 units of starting value, the terminal value would be `2500 * 1.12 = 2800`. Use the actual simulated endings to calculate their mean and median.

In [ ]:
endings = pd.Series([1.0, 1.1, 1.2, 3.0])
low, high = endings.quantile([0.025, 0.975])
display(
    pd.Series(
        {
            "mean": endings.mean(),
            "median": endings.median(),
            "midpoint_of_quantiles": (low + high) / 2,
        }
    )
)
assert not np.isclose(endings.mean(), (low + high) / 2)

A mean, a median, and a midpoint of quantiles answer different questions. The scenario quantiles also do not measure the model's accuracy. The code walkthrough and regression tests show how these distinctions are checked.